In [ ]:
!pip install --upgrade transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 40.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3


In [ ]:
!pip install snorkel


In [ ]:
!pip install snorkel transformers spacy
!python -m spacy download en_core_web_sm  # Ensure SpaCy model is installed


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 833.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import re
import pandas as pd
import spacy
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from transformers import pipeline, AutoTokenizer
from snorkel.labeling.model import LabelModel


In [ ]:
ABSTAIN = -1
POSITIVE = 1
NEGATIVE = 0

In [ ]:
df = pd.read_csv("movie.csv", on_bad_lines="skip", quoting=3, encoding="utf-8")
df = df.dropna(subset=["text"])
print(df.head())

                                                                                                                                                    text  \
"I grew up (b. 1965) watching and loving the Th... during lunch and after school. We all wanted to...   his version was completely hopeless. A waste ...   
"Even though I have great interest in Biblical ... I was bored to death every minute of the movie....   the acting is most of the time a Joke and the...   
"If you want a fun romp with loads of subtle humor then you will enjoy this flick.<br /><br />I do...   but it is well done. Ericka Eleniak is absolu...   
"SUcks. That's all I got to say about this sorr... what the hell were they thinking? The idiots in...                                                  0   
"I love this movie ! I think I've seen it 5 tim... it's a thriller and there is great tension. But...                                                  1   

                                                               

In [ ]:
nlp = spacy.load("en_core_web_sm")


In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis",
                              model="distilbert-base-uncased-finetuned-sst-2-english",
                              device=0)  # Use GPU


Device set to use cuda:0


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


In [ ]:
# Expanded Negative Words LF
@labeling_function()
def lf_strong_negative_words(row):
    text = str(row.text).lower()
    negative_words = ["awful", "horrendous", "worst", "sucks", "disgusting", "hate", "waste", "boring"]
    return NEGATIVE if any(word in text for word in negative_words) else ABSTAIN

# Pattern-based Sentiment Detection: Exclamation Marks
@labeling_function()
def lf_exclamation_marks(row):
    text = str(row.text)
    return POSITIVE if "!!!" in text else ABSTAIN

# Pattern-based Sentiment Detection: Negations
@labeling_function()
def lf_repeated_negation(row):
    text = str(row.text).lower()
    if re.search(r"not good|not worth|not recommend|never again", text):
        return NEGATIVE
    return ABSTAIN

In [ ]:
@labeling_function()
def lf_transformer_sentiment(row):
    text = str(row.text)

    # Properly tokenize and truncate input before passing to model
    encoded_text = tokenizer(text, truncation=True, max_length=512, return_tensors="pt")

    # Decode properly truncated text before sentiment analysis
    truncated_text = tokenizer.decode(encoded_text["input_ids"][0], skip_special_tokens=True)

    # Run sentiment analysis
    result = sentiment_pipeline(truncated_text)[0]

    return POSITIVE if result["label"] == "POSITIVE" else NEGATIVE


In [ ]:
@labeling_function()
def lf_spacy_sentiment(row):
    text = str(row.text)
    doc = nlp(text)
    sentiment_score = sum(token.sentiment for token in doc) / len(doc)
    return POSITIVE if sentiment_score > 0 else NEGATIVE


In [ ]:
lfs = [
    lf_strong_negative_words,
    lf_exclamation_marks,
    lf_repeated_negation,
    lf_transformer_sentiment,
    lf_spacy_sentiment
]


In [ ]:
applier = PandasLFApplier(lfs=lfs)
label_matrix = applier.apply(df[["text"]])


100%|██████████| 4419/4419 [01:18<00:00, 55.98it/s]


In [ ]:
LFAnalysis(label_matrix, lfs).lf_summary()


,j,Polarity,Coverage,Overlaps,Conflicts
lf_strong_negative_words,0,[0],0.054990,0.054990,0.008826
lf_exclamation_marks,1,[1],0.019688,0.019688,0.019688
lf_repeated_negation,2,[0],0.004752,0.004752,0.000226
lf_transformer_sentiment,3,"[0, 1]",1.000000,1.000000,0.516180
lf_spacy_sentiment,4,[0],1.000000,1.000000,0.516180


In [ ]:
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(label_matrix, n_epochs=500, log_freq=100, seed=42)

100%|██████████| 500/500 [00:01<00:00, 452.31epoch/s]


In [ ]:
df["snorkel_label"] = label_model.predict(label_matrix)


In [ ]:
df.to_csv("snorkel_weakly_labeled.csv", index=False)
print("✅ Snorkel labeling complete! Labeled dataset saved.")


✅ Snorkel labeling complete! Labeled dataset saved.
